In [ ]:
%%capture
!pip install jsonlines
!pip install gdown

In [ ]:
!gdown https://drive.google.com/uc?id=1i6tav-kFn3InbXHcgN6HkkB486LoTHQk

In [ ]:
!unzip DARIJA-SPEECH.zip

In [ ]:
import os
import jsonlines
import pandas as pd
data = []
with jsonlines.open("DARIJA/metadata.json", "r") as reader:
  for obj in reader:
    data.append(obj)
df = pd.DataFrame(data)
df.head(2)

df["file_name"] = "DARIJA/wavs/" + df["file_name"] + ".wav"
df["exists"] = df["file_name"].map(lambda x : os.path.exists(x))

In [ ]:
!pip install --upgrade pyarrow


In [ ]:
!pip install --upgrade datasets


In [ ]:
import os
#os.kill(os.getpid(), 9)


In [ ]:
from datasets import Dataset
audio_folder = "/content/DARIJA/wavs"
from datasets import Dataset
from datasets import Audio

audio_paths = [os.path.join(audio_folder, file) for file in os.listdir(audio_folder)]
dataset = Dataset.from_dict({"audio": audio_paths}).cast_column("audio", Audio())

In [ ]:
import os
import jsonlines
import pandas as pd
data = []
with jsonlines.open("/content/DARIJA/metadata.json", "r") as reader:
  for obj in reader:
    data.append(obj)
df = pd.DataFrame(data)
df.head(2)

df["file_name"] = "DARIJA/wavs/" + df["file_name"] + ".wav"
df["exists"] = df["file_name"].map(lambda x : os.path.exists(x))

In [ ]:
df.head()

In [ ]:
audio_dataset= dataset
transcriptions = []

# Itérer sur les chemins des fichiers audio existants dans le dataset
for path in audio_dataset["audio"]:
    # Extraire le nom de fichier sans extension
    file_name = os.path.basename(path["path"]).split('.')[0]

    # Trouver la transcription correspondante dans le DataFrame
    row = df[df['file_name'].str.contains(file_name)]

    if not row.empty:
        # Ajouter la transcription trouvée
         transcriptions.append(str(row['sentence_whisper'].values[0]))
    else:
        # Ajouter une transcription vide si aucune correspondance n'est trouvée
        transcriptions.append("")

# Ajouter la colonne "text" au dataset existant
audio_dataset = audio_dataset.add_column("text", transcriptions)

# Afficher quelques exemples pour vérifier la structure
print(audio_dataset[0])

In [ ]:
# Filtrer les enregistrements où 'text' n'est pas NaN
audio_dataset = audio_dataset.filter(lambda x: x['text'] is not None and str(x['text']).strip().lower() != 'nan')

print(audio_dataset)


In [ ]:
import re

# Expression régulière pour détecter les caractères arabes, les espaces et les ponctuations courantes
arabic_pattern = re.compile(r"^[\u0600-\u06FF\s]+$")

# Fonction pour trouver les enregistrements avec des caractères non-arabes
def detect_unusual_characters(text):
    if not arabic_pattern.match(text):
        return text
    return None

# Filtrer les enregistrements avec des caractères non-arabes
unusual_texts = audio_dataset.filter(lambda x: detect_unusual_characters(x['text']) is not None)

# Afficher les textes contenant des caractères non-arabes
for text in unusual_texts['text']:
    print(text)


In [ ]:
from datasets import Dataset
import re

# Dictionnaire de correspondance des chiffres occidentaux aux chiffres arabes
chiffres_arabes = {
    '0': '٠',
    '1': '١',
    '2': '٢',
    '3': '٣',
    '4': '٤',
    '5': '٥',
    '6': '٦',
    '7': '٧',
    '8': '٨',
    '9': '٩'
}

# Fonction pour transformer les chiffres occidentaux en chiffres arabes
def transformer_chiffres_arabes(text):
    for occidental, arabe in chiffres_arabes.items():
        text = re.sub(f'{occidental}', arabe, text)
    return text

# Fonction à appliquer sur chaque exemple du dataset
def remplacer_chiffres(examples):
    examples['text'] = transformer_chiffres_arabes(examples['text'])
    return examples

# Supposons que ton dataset s'appelle audio_dataset
# Appliquer la transformation sur tout le dataset
audio_dataset = audio_dataset.map(remplacer_chiffres)

# Afficher quelques exemples pour vérifier la transformation
print(audio_dataset['text'])


In [ ]:
#!pip install datasets soundfile speechbrain
#from datasets import Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

csv_file = "/content/drive/MyDrive/speechT5/code/transcript.csv"
data = pd.read_csv(csv_file, sep=",")
print(data.head())



In [ ]:
data['path'] = data['path'].str.replace(r'^downs_clips\\', '', regex=True)



In [ ]:
audio_dataset=Dataset.from_dict({"audio" : [], "text" : []})

In [ ]:

from IPython.display import Audio
Audio("/content/DARIJA/wavs/DAR-0.wav")

In [ ]:
!pip install pydub
import os
from pydub import AudioSegment
audio_folder = "/content/drive/MyDrive/speechT5/code/wavs"  # Remplacez par le chemin de votre dossier contenant les fichiers wav

# Assurez-vous que les fichiers audio sont au format wav et pas mp3
audio_paths = [os.path.join(audio_folder, f) for f in os.listdir(audio_folder) if f.endswith('.wav')]

# Fonction pour charger les fichiers audio en tant qu'objets AudioSegment
def load_audio(file_path):
    return AudioSegment.from_file(file_path)

# Charger les fichiers audio en objets AudioSegment
audio_data = [load_audio(path) for path in audio_paths]

# Créer une liste pour les transcriptions (en supposant qu'elles sont alignées avec les fichiers audio par nom de fichier)
transcriptions = []
speaker_ids = []

for path in audio_paths:
    # Extraire le nom de fichier sans extension
    file_name = os.path.basename(path).split('.')[0]

    # Trouver la transcription et le speaker_id correspondant dans le DataFrame
    row = data[data['path'].str.contains(file_name)]

    if not row.empty:
        transcriptions.append(row['sentence'].values[0])
        speaker_ids.append(row['client_id'].values[0])  # Assurez-vous que la colonne 'client_id' existe dans votre DataFrame
    else:
        transcriptions.append("")  # ou traiter l'audio sans transcription correspondante comme vous le souhaitez
        speaker_ids.append("")  # ou un identifiant de locuteur par défaut

# Assurez-vous que les listes audio_data, transcriptions et speaker_ids sont de même longueur
assert len(audio_data) == len(transcriptions) == len(speaker_ids)

# Créer un dictionnaire pour le Dataset
dataset_dict = {
    "audio": audio_paths,  # ou directement les objets audio_data si vous les convertissez au format attendu par Hugging Face
    "text": transcriptions,
    "speaker_id": speaker_ids
}

# Créer le Dataset
audio_dataset = Dataset.from_dict(dataset_dict)

# Afficher un échantillon pour vérifier
print(audio_dataset[0])


In [ ]:
!pip install soundfile numpy


In [ ]:
import soundfile as sf
import numpy as np
from datasets import Dataset


In [ ]:
def read_audio_file(file_path):
    audio_data, sample_rate = sf.read(file_path)
    return audio_data, sample_rate
def convert_audio_to_array(example):
    audio_path = example['audio']
    audio_array, sample_rate = read_audio_file(audio_path)
    return {
        "audio": {
            "path": audio_path,
            "array": audio_array,
            "sampling_rate": sample_rate
        }
    }

# Mettre à jour le dataset avec les tableaux numpy
audio_dataset = audio_dataset.map(convert_audio_to_array, remove_columns=['audio'])

# Afficher le dataset pour vérifier
print(audio_dataset[0])

In [ ]:
print(audio_dataset[0])

In [ ]:
"""from datasets import load_dataset, Audio

dataset = load_dataset(
    "facebook/voxpopuli", "nl", split="train"
)"""

In [ ]:
#print(dataset[0])

In [ ]:
!pip install transformers

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech

#processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
#model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
processor = SpeechT5Processor.from_pretrained("MBZUAI/speecht5_tts_clartts_ar")
model = SpeechT5ForTextToSpeech.from_pretrained("MBZUAI/speecht5_tts_clartts_ar")

In [ ]:
tokenizer = processor.tokenizer

In [ ]:
from collections import defaultdict
speaker_counts = defaultdict(int)

for speaker_id in audio_dataset["speaker_id"]:
    speaker_counts[speaker_id] += 1

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(speaker_counts.values(), bins=50)
plt.ylabel("Speakers")
plt.xlabel("Examples")
plt.show()

In [ ]:
len(audio_dataset)

In [ ]:
def select_speaker(speaker_id):
    return 100 <= speaker_counts[speaker_id]

audio_dataset = audio_dataset.filter(select_speaker, input_columns=["speaker_id"])

In [ ]:
#!pip uninstall speechbrain -y
!pip install speechbrain==0.5.16

import os
import torch
from speechbrain.pretrained import EncoderClassifier

# Spécifier le modèle et le dispositif
spk_model_name = "speechbrain/spkrec-xvect-voxceleb"
device = "cuda" if torch.cuda.is_available() else "cpu"


# Définir le chemin de sauvegarde dans Google Drive
savedir = os.path.join("/content/drive/My Drive/speechT5/code/speechbrain_models", spk_model_name)
# Charger le modèle avec le chemin de sauvegarde spécifié
speaker_model = EncoderClassifier.from_hparams(
    source=spk_model_name,
    savedir=savedir
)

# Définir le dispositif après le chargement du modèle
#speaker_model.modules = speaker_model.modules.to(device)
# Fonction pour créer les embeddings de locuteur
def create_speaker_embedding(waveform):
    with torch.no_grad():
        speaker_embeddings = speaker_model.encode_batch(torch.tensor(waveform))
        speaker_embeddings = torch.nn.functional.normalize(speaker_embeddings, dim=2)
        speaker_embeddings = speaker_embeddings.squeeze().cpu().numpy()
    return speaker_embeddings


In [ ]:
def prepare_dataset(example):
    # load the audio data; if necessary, this resamples the audio to 16kHz
    audio = example["audio"]

    # feature extraction and tokenization
    example = processor(
        text=example["text"],
        audio_target=audio["array"],
        return_attention_mask=False,
    )

    # strip off the batch dimension
    example["labels"] = example["labels"][0]

    # use SpeechBrain to obtain x-vector
    example["speaker_embeddings"] = create_speaker_embedding(audio["array"])

    return example

In [ ]:
processed_example = prepare_dataset(audio_dataset[0])

In [ ]:
from transformers import SpeechT5HifiGan
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

In [ ]:
spectrogram = torch.tensor(processed_example["labels"])
with torch.no_grad():
    speech = vocoder(spectrogram)

In [ ]:
list(processed_example.keys())

In [ ]:
tokenizer.decode(processed_example["input_ids"])

In [ ]:
processed_example["speaker_embeddings"].shape

In [ ]:
from IPython.display import Audio
Audio(speech.cpu().numpy(), rate=16000)

In [ ]:
dataset = audio_dataset.map(
    prepare_dataset, remove_columns=audio_dataset.column_names,
)

In [ ]:
def is_not_too_long(input_ids):
    input_length = len(input_ids)
    return input_length < 200

dataset = dataset.filter(is_not_too_long, input_columns=["input_ids"])
print(len(dataset))

In [ ]:
dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
dataset

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class TTSDataCollatorWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:

        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        label_features = [{"input_values": feature["labels"]} for feature in features]
        speaker_features = [feature["speaker_embeddings"] for feature in features]

        # collate the inputs and targets into a batch
        batch = processor.pad(
            input_ids=input_ids,
            labels=label_features,
            return_tensors="pt",
        )

        # replace padding with -100 to ignore loss correctly
        batch["labels"] = batch["labels"].masked_fill(
            batch.decoder_attention_mask.unsqueeze(-1).ne(1), -100
        )

        # not used during fine-tuning
        del batch["decoder_attention_mask"]

        # round down target lengths to multiple of reduction factor
        if model.config.reduction_factor > 1:
            target_lengths = torch.tensor([
                len(feature["input_values"]) for feature in label_features
            ])
            target_lengths = target_lengths.new([
                length - length % model.config.reduction_factor for length in target_lengths
            ])
            max_length = max(target_lengths)
            batch["labels"] = batch["labels"][:, :max_length]

        # also add in the speaker embeddings
        batch["speaker_embeddings"] = torch.tensor(speaker_features)

        return batch

In [ ]:
data_collator = TTSDataCollatorWithPadding(processor=processor)

In [ ]:
features = [
    dataset["train"][0],
    dataset["train"][1],
    dataset["train"][20],
]

batch = data_collator(features)

In [ ]:
{k:v.shape for k,v in batch.items()}

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.config.use_cache = False

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/My Drive/speechT5/code/transformer",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    greater_is_better=False,
    label_names=["labels"],
    push_to_hub=False,
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    tokenizer=processor.tokenizer,)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("/content/drive/My Drive/speechT5/code/transform")
processor.save_pretrained("/content/drive/My Drive/speechT5/code/transform")


In [ ]:
model = SpeechT5ForTextToSpeech.from_pretrained("/content/drive/My Drive/speechT5/code/transform")
processor = SpeechT5Processor.from_pretrained("/content/drive/My Drive/speechT5/code/transform")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")


In [ ]:
# Exemple de texte à convertir en parole
text = "Ina ƙaunar ka, ina sonki. Za ki zama matata"

# Charger un exemple d'embeddings de locuteur depuis le dataset
example = dataset["test"][102]  # Assurez-vous que l'index est valide
speaker_embeddings = torch.tensor(example["speaker_embeddings"]).unsqueeze(0)


In [ ]:
# Préparer les entrées pour le modèle
inputs = processor(text=text, return_tensors="pt")

# Générer le spectrogramme
with torch.no_grad():
    spectrogram = model.generate_speech(inputs["input_ids"], speaker_embeddings)


In [ ]:
# Synthétiser l'onde sonore à partir du spectrogramme
with torch.no_grad():
    speech = vocoder(spectrogram)


In [ ]:
Audio(speech.numpy(), rate=16000)


In [ ]:
from pydub import AudioSegment
rate = 16000
speech_array = speech.numpy()

# Convertir le tableau NumPy en AudioSegment
audio = AudioSegment(
    speech_array.tobytes(),
    frame_rate=rate,
    sample_width=speech_array.dtype.itemsize,
    channels=1
)

# Enregistrer le fichier audio
output_file = "test_audio.wav"
audio.export(output_file, format="wav")

print(f"Audio enregistré sous {output_file}")
